<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026_0501psy3a_lect03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 心理学特講IIIA 第3回：Simon課題

**担当**: 浅川伸一  
**2026年度 前期**

---

## 本日の実習内容

1. Simon効果の理論
2. Simon課題の設計
3. 模擬データの生成と分析
4. 記述統計と可視化
5. 対応のあるt検定
6. Stroop課題との比較
7. 練習効果の分析

---
## 1. Simon効果とは

### 1.1 背景

**J. Richard Simon (1969)**  
刺激の空間的位置と反応の空間的位置の対応関係が、反応時間に影響を与えることを発見。

### 1.2 基本的な設定

- **課題**: 刺激の色に応じてキーを押す
- **刺激**: 左または右に提示される色（赤・青）
- **反応**: 左または右のキー押し
- **重要**: 刺激の位置は課題に無関連

### 1.3 Simon効果

- **一致条件**: 刺激位置と反応位置が同じ側 → 速い
- **不一致条件**: 刺激位置と反応位置が反対側 → 遅い
- **Simon効果** = 不一致RT - 一致RT（通常50-100ms）

---
## 2. 環境準備

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel, ttest_ind

# 日本語フォントの設定
!pip install japanize-matplotlib --quiet
import japanize_matplotlib

# 警告を非表示
import warnings
warnings.filterwarnings('ignore')

# 乱数シードの固定（再現性のため）
np.random.seed(42)

print("ライブラリの読み込み完了")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 25.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ライブラリの読み込み完了


---
## 3. Simon課題の設計

### 3.1 刺激と反応の定義

In [ ]:
# Simon課題の刺激定義
simon_stimuli = {
    'congruent': [
        {'color': 'red', 'position': 'left', 'response': 'left', 'color_jp': '赤'},
        {'color': 'blue', 'position': 'right', 'response': 'right', 'color_jp': '青'},
        {'color': 'red', 'position': 'right', 'response': 'right', 'color_jp': '赤'},
        {'color': 'blue', 'position': 'left', 'response': 'left', 'color_jp': '青'}
    ],
    'incongruent': [
        {'color': 'red', 'position': 'right', 'response': 'left', 'color_jp': '赤'},
        {'color': 'blue', 'position': 'left', 'response': 'right', 'color_jp': '青'},
        {'color': 'red', 'position': 'left', 'response': 'right', 'color_jp': '赤'},
        {'color': 'blue', 'position': 'right', 'response': 'left', 'color_jp': '青'}
    ]
}

# 刺激の確認
print("=== Simon課題の刺激確認 ===")
print("\n一致条件の例:")
stim = simon_stimuli['congruent'][0]
print(f"  色: {stim['color_jp']}({stim['color']})")
print(f"  位置: {stim['position']}")
print(f"  正解反応: {stim['response']}")
print(f"  → 位置と反応が一致")

print("\n不一致条件の例:")
stim = simon_stimuli['incongruent'][0]
print(f"  色: {stim['color_jp']}({stim['color']})")
print(f"  位置: {stim['position']}")
print(f"  正解反応: {stim['response']}")
print(f"  → 位置と反応が不一致")

---
## 4. 模擬データの生成

### 4.1 データ生成関数

In [ ]:
def generate_simon_data(n_trials_per_condition=40, participant_id=1):
    """
    Simon課題の模擬データを生成する関数

    Parameters:
    -----------
    n_trials_per_condition : int
        各条件での試行数（デフォルト: 40）
    participant_id : int
        参加者ID（デフォルト: 1）

    Returns:
    --------
    df : pandas.DataFrame
        生成されたデータ
    """

    data = []

    # 各条件でのベース反応時間（ミリ秒）
    # Simon効果はStroopより小さい（50-100ms程度）
    rt_parameters = {
        'congruent': {'mean': 550, 'std': 70},      # 一致条件: 速い
        'incongruent': {'mean': 650, 'std': 80}     # 不一致条件: 遅い
    }

    # 正答率（エラー率）
    error_rate = {
        'congruent': 0.02,      # 2%
        'incongruent': 0.05     # 5%
    }

    trial_counter = 1

    for condition in ['congruent', 'incongruent']:
        params = rt_parameters[condition]

        for trial in range(n_trials_per_condition):
            # 反応時間の生成（正規分布）
            rt = np.random.normal(params['mean'], params['std'])
            rt = max(300, rt)  # 最小値を300msに設定

            # 正誤の生成
            accuracy = 0 if np.random.random() < error_rate[condition] else 1

            # エラー試行は反応時間を調整（エラーは速い傾向）
            if accuracy == 0:
                rt = rt * 0.85

            data.append({
                'participant': participant_id,
                'trial': trial_counter,
                'condition': condition,
                'rt': round(rt, 2),
                'accuracy': accuracy
            })

            trial_counter += 1

    df = pd.DataFrame(data)

    # 条件をカテゴリ型に変換（順序を保持）
    df['condition'] = pd.Categorical(
        df['condition'],
        categories=['congruent', 'incongruent'],
        ordered=True
    )

    return df

# データ生成
df_simon = generate_simon_data(n_trials_per_condition=40)

print(f"生成されたデータ: {len(df_simon)}試行")
print(f"\n最初の10試行:")
df_simon.head(10)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_simon)

### 4.2 データの確認

In [ ]:
# データの基本情報
print("=== データの基本情報 ===")
print(df_simon.info())

print("\n=== 条件ごとの試行数 ===")
print(df_simon['condition'].value_counts().sort_index())

print("\n=== 記述統計 ===")
print(df_simon.describe())

---
## 5. 記述統計

### 5.1 条件別の基本統計量

In [ ]:
# 正答試行のみを抽出
df_simon_correct = df_simon[df_simon['accuracy'] == 1].copy()

print(f"全試行数: {len(df_simon)}")
print(f"正答試行数: {len(df_simon_correct)}")
print(f"全体正答率: {len(df_simon_correct)/len(df_simon)*100:.2f}%")

# 条件別の記述統計
summary = df_simon_correct.groupby('condition')['rt'].agg([
    ('試行数', 'count'),
    ('平均', 'mean'),
    ('標準偏差', 'std'),
    ('最小値', 'min'),
    ('最大値', 'max'),
    ('中央値', 'median')
]).round(2)

print("\n=== 条件別反応時間の記述統計（正答試行のみ）===")
print(summary)

# Simon効果の計算
mean_congruent = df_simon_correct[df_simon_correct['condition'] == 'congruent']['rt'].mean()
mean_incongruent = df_simon_correct[df_simon_correct['condition'] == 'incongruent']['rt'].mean()
simon_effect = mean_incongruent - mean_congruent

print(f"\n=== Simon効果 ===")
print(f"一致条件の平均RT: {mean_congruent:.2f} ms")
print(f"不一致条件の平均RT: {mean_incongruent:.2f} ms")
print(f"Simon効果: {simon_effect:.2f} ms")

### 5.2 条件別正答率

In [ ]:
# 条件別正答率
accuracy_by_condition = df_simon.groupby('condition')['accuracy'].agg([
    ('正答数', 'sum'),
    ('試行数', 'count'),
    ('正答率', 'mean')
])

accuracy_by_condition['正答率(%)'] = (accuracy_by_condition['正答率'] * 100).round(2)

print("=== 条件別正答率 ===")
print(accuracy_by_condition[['正答数', '試行数', '正答率(%)']])

---
## 6. データ可視化

### 6.1 箱ひげ図

In [ ]:
# 箱ひげ図
plt.figure(figsize=(10, 6))
bp = df_simon_correct.boxplot(column='rt', by='condition', figsize=(10, 6), patch_artist=True)
plt.title('Simon課題：条件別反応時間の分布', fontsize=14)
plt.suptitle('')  # デフォルトタイトルを削除
plt.xlabel('条件', fontsize=12)
plt.ylabel('反応時間（ms）', fontsize=12)
plt.xticks([1, 2], ['一致', '不一致'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 バイオリンプロット

In [ ]:
# バイオリンプロット
plt.figure(figsize=(10, 6))
sns.violinplot(data=df_simon_correct, x='condition', y='rt', palette='Set2')
plt.title('Simon課題：条件別反応時間の分布（バイオリンプロット）', fontsize=14)
plt.xlabel('条件', fontsize=12)
plt.ylabel('反応時間（ms）', fontsize=12)
plt.xticks([0, 1], ['一致', '不一致'])
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 6.3 平均値と標準誤差

In [ ]:
# 平均値と標準誤差のバープロット
means = df_simon_correct.groupby('condition')['rt'].mean()
sems = df_simon_correct.groupby('condition')['rt'].sem()  # 標準誤差

plt.figure(figsize=(10, 6))
x_pos = np.arange(len(means))
plt.bar(x_pos, means.values, yerr=sems.values, capsize=10,
        alpha=0.7, color=['#66c2a5', '#fc8d62'])
plt.xlabel('条件', fontsize=12)
plt.ylabel('平均反応時間（ms）', fontsize=12)
plt.title('Simon課題：条件別平均反応時間（エラーバー = 標準誤差）', fontsize=14)
plt.xticks(x_pos, ['一致', '不一致'])
plt.ylim(0, max(means.values) * 1.3)
plt.grid(True, alpha=0.3, axis='y')

# Simon効果を示す線を追加
y_max = max(means.values) * 1.15
plt.plot([0, 1], [y_max, y_max], 'k-', linewidth=1.5)
plt.text(0.5, y_max * 1.02, f'Simon効果: {simon_effect:.1f}ms',
         ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# 数値を表示
print("=== 平均反応時間（標準誤差）===")
for condition, mean, sem in zip(['一致', '不一致'], means.values, sems.values):
    print(f"{condition}条件: {mean:.2f} ms (SE = {sem:.2f})")

---
## 7. 統計的検定

### 7.1 対応のあるt検定（Paired t-test）

Simon課題は参加者内計画なので、対応のあるt検定を使用します。

**帰無仮説（H₀）**: 一致条件と不一致条件の平均反応時間に差がない  
**対立仮説（H₁）**: 不一致条件の平均反応時間の方が長い

In [ ]:
# 条件ごとのデータを抽出
congruent_rt = df_simon_correct[df_simon_correct['condition'] == 'congruent']['rt'].values
incongruent_rt = df_simon_correct[df_simon_correct['condition'] == 'incongruent']['rt'].values

# # 対応のある t 検定
# t_statistic, p_value = ttest_rel(incongruent_rt, congruent_rt)

# 対応のない t 検定
t_statistic, p_value = ttest_ind(incongruent_rt, congruent_rt)

# print("=== 対応のあるt検定（Paired t-test）===")
print("=== 対応のない t 検定（Paired t-test）===")
print(f"t値: {t_statistic:.4f}")
print(f"p値: {p_value:.6f}")
print(f"自由度: {len(congruent_rt) - 1}")
print(f"\n結果: 有意水準5%で{'有意' if p_value < 0.05 else '有意でない'}")

if p_value < 0.05:
    print("→ 不一致条件の反応時間が有意に長い")
    print("→ Simon効果が確認された")
else:
    print("→ 条件間で反応時間に有意な差は認められない")

### 7.2 効果量（Cohen's d）

In [ ]:
try:
    import pingouin as pg
except ImportError:
    !pip install pingouin
    import pingouin as pg

# # Hedges' g の計算
# g = pg.compute_effsize(x, y, eftype='hedges')
# # Glass's delta の計算
# delta = pg.compute_effsize(x, y, eftype='glass')


In [ ]:
def cohens_d_paired(group1, group2):
    """
    対応のあるデータのCohen's dを計算
    """
    diff = group1 - group2
    return np.mean(diff) / np.std(diff, ddof=1)

# 効果量の計算
# d = cohens_d_paired(incongruent_rt, congruent_rt)

x, y = incongruent_rt, congruent_rt

# Hedges' g の計算
g = pg.compute_effsize(x, y, eftype='hedges')
# Glass's delta の計算
# delta = pg.compute_effsize(x, y, eftype='glass')
print(f'g:{g}')
# print(f'delta:{delta}')
d = g

#print(f"効果量（Cohen's d）: {d:.4f}")
print("\n効果量の目安（Cohen, 1988）:")
print("  小: d = 0.2")
print("  中: d = 0.5")
print("  大: d = 0.8")

if abs(d) < 0.5:
    effect_size = "小"
elif abs(d) < 0.8:
    effect_size = "中"
else:
    effect_size = "大"

print(f"\n今回の効果量は「{effect_size}」に相当")

---
## 8. Stroop課題との比較

### 8.1 Stroopデータの生成（前回のデータを再現）

In [ ]:
def generate_stroop_data(n_trials=30):
    """
    Stroop課題の模擬データ生成（前回の復習）
    """
    data = []

    rt_base = {
        'congruent': 600,
        'neutral': 700,
        'incongruent': 850
    }

    for condition in ['congruent', 'neutral', 'incongruent']:
        for trial in range(n_trials):
            rt = np.random.normal(rt_base[condition], 100)
            accuracy = 1 if np.random.random() > 0.05 else 0

            data.append({
                'condition': condition,
                'rt': max(300, rt),
                'accuracy': accuracy
            })

    return pd.DataFrame(data)

# Stroopデータの生成
df_stroop = generate_stroop_data(n_trials=30)
df_stroop_correct = df_stroop[df_stroop['accuracy'] == 1]

print("=== Stroopデータ生成完了 ===")
print(f"試行数: {len(df_stroop_correct)}")

### 8.2 干渉効果の比較

In [ ]:
# Stroop効果の計算
stroop_congruent = df_stroop_correct[df_stroop_correct['condition'] == 'congruent']['rt'].mean()
stroop_incongruent = df_stroop_correct[df_stroop_correct['condition'] == 'incongruent']['rt'].mean()
stroop_effect = stroop_incongruent - stroop_congruent

# Simon効果（再掲）
simon_congruent = df_simon_correct[df_simon_correct['condition'] == 'congruent']['rt'].mean()
simon_incongruent = df_simon_correct[df_simon_correct['condition'] == 'incongruent']['rt'].mean()
simon_effect_calc = simon_incongruent - simon_congruent

# 比較表の作成
comparison = pd.DataFrame({
    '課題': ['Stroop', 'Simon'],
    '一致条件(ms)': [stroop_congruent, simon_congruent],
    '不一致条件(ms)': [stroop_incongruent, simon_incongruent],
    '干渉効果(ms)': [stroop_effect, simon_effect_calc]
})

print("=== StroopとSimonの干渉効果比較 ===")
print(comparison.round(2))
print(f"\nStroop効果の方が約{(stroop_effect - simon_effect_calc):.1f}ms大きい")

### 8.3 可視化による比較

In [ ]:
# 両課題の干渉効果を比較するグラフ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 各課題の条件別平均RT
tasks = ['Stroop', 'Simon']
congruent_means = [stroop_congruent, simon_congruent]
incongruent_means = [stroop_incongruent, simon_incongruent]

x = np.arange(len(tasks))
width = 0.35

axes[0].bar(x - width/2, congruent_means, width, label='一致', alpha=0.7, color='#66c2a5')
axes[0].bar(x + width/2, incongruent_means, width, label='不一致', alpha=0.7, color='#fc8d62')
axes[0].set_xlabel('課題', fontsize=12)
axes[0].set_ylabel('平均反応時間（ms）', fontsize=12)
axes[0].set_title('課題別・条件別平均反応時間', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tasks)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# 右: 干渉効果の大きさ
effects = [stroop_effect, simon_effect_calc]
colors_effect = ['#e74c3c', '#3498db']

axes[1].bar(tasks, effects, alpha=0.7, color=colors_effect)
axes[1].set_xlabel('課題', fontsize=12)
axes[1].set_ylabel('干渉効果（ms）', fontsize=12)
axes[1].set_title('干渉効果の大きさ比較', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

# 数値をバーの上に表示
for i, (task, effect) in enumerate(zip(tasks, effects)):
    axes[1].text(i, effect + 10, f'{effect:.1f}ms',
                ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### 8.4 なぜ効果量が異なるのか？

**Stroop効果の方が大きい理由**

1. **干渉の性質**
   - Stroop: 意味的・言語的干渉（深い処理レベル）
   - Simon: 空間的干渉（知覚レベル）

2. **自動性の強さ**
   - 文字の読み取りは高度に自動化
   - 空間位置の符号化も自動的だが、言語ほどではない

3. **処理の深さ**
   - Stroop: 意味処理まで到達
   - Simon: 主に知覚・運動レベル

**共通点**
- どちらも課題無関連情報が自動的に処理される
- 反応選択プロセスに影響

---
## 9. 練習効果の分析

### 9.1 試行ブロックによる変化

In [ ]:
# 試行をブロックに分割（前半・後半）
df_simon_correct['block'] = df_simon_correct.groupby('condition').cumcount() < 20
df_simon_correct['block'] = df_simon_correct['block'].map({True: '前半', False: '後半'})

# ブロック×条件の平均反応時間
block_summary = df_simon_correct.pivot_table(
    values='rt',
    index='condition',
    columns='block',
    aggfunc='mean'
)

print("=== ブロック別・条件別平均反応時間 ===")
print(block_summary.round(2))

# 可視化
block_summary.plot(kind='bar', figsize=(10, 6), rot=0)
plt.title('Simon課題：ブロック別・条件別平均反応時間', fontsize=14)
plt.xlabel('条件', fontsize=12)
plt.ylabel('平均反応時間（ms）', fontsize=12)
plt.xticks([0, 1], ['一致', '不一致'])
plt.legend(title='ブロック')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Simon効果のブロック別比較
simon_effect_early = block_summary.loc['incongruent', '前半'] - block_summary.loc['congruent', '前半']
simon_effect_late = block_summary.loc['incongruent', '後半'] - block_summary.loc['congruent', '後半']

print(f"\n前半のSimon効果: {simon_effect_early:.2f} ms")
print(f"後半のSimon効果: {simon_effect_late:.2f} ms")
print(f"変化: {simon_effect_late - simon_effect_early:.2f} ms")

---
## 10. 結果の解釈

### 10.1 主な発見

In [ ]:
print("=== Simon課題の結果まとめ ===")
print("\n1. 条件別平均反応時間:")
for condition, label in zip(['congruent', 'incongruent'], ['一致', '不一致']):
    mean = df_simon_correct[df_simon_correct['condition'] == condition]['rt'].mean()
    print(f"   {label}条件: {mean:.2f} ms")

print(f"\n2. Simon効果: {simon_effect:.2f} ms")

print("\n3. 統計的検定:")
print(f"   対応のあるt検定: t({len(congruent_rt)-1}) = {t_statistic:.4f}, p = {p_value:.6f}")
print(f"   効果量（Cohen's d）: {d:.4f}")

print("\n4. Stroopとの比較:")
print(f"   Stroop効果: {stroop_effect:.2f} ms")
print(f"   Simon効果: {simon_effect:.2f} ms")
print(f"   差: {stroop_effect - simon_effect:.2f} ms")

print("\n5. 解釈:")
print("   - 典型的なSimon効果が確認された")
print("   - 不一致条件で反応時間が有意に長い")
print("   - 空間的一致性が反応時間に影響")
print("   - Stroopより効果は小さいが明確に存在")

### 10.2 理論的意義

**刺激-反応適合性（S-R Compatibility）**

1. **自動的な空間コーディング**
   - 刺激の位置は課題に無関連でも符号化される
   - 意図的に無視することが困難

2. **反応選択プロセス**
   - 刺激位置が反応側を自動的に活性化
   - 不一致時は競合を解決する必要

3. **応用可能性**
   - インターフェース設計（ボタン配置）
   - 交通信号の位置と反応
   - ヒューマンエラーの予防

---
## 11. 準備学習（次回までの課題）

以下の課題に取り組んでください。

### 課題1（100字程度）
Simon効果について、刺激-反応適合性の観点から説明してください。

### 課題2
日常生活でSimon効果が見られる例を1つ挙げてください。

### 課題3（50字程度）
なぜStroop効果の方がSimon効果より大きいのか、説明してください。

---

### 解答欄（この下に記入してください）

**課題1（Simon効果の説明）**:



**課題2（日常生活の例）**:



**課題3（StroopとSimonの違い）**:




---
## 参考文献

1. Simon, J. R. (1969). Reactions toward the source of stimulation. *Journal of Experimental Psychology, 81*(1), 174-176.

2. Lu, C. H., & Proctor, R. W. (1995). The influence of irrelevant location information on performance: A review of the Simon and spatial Stroop effects. *Psychonomic Bulletin & Review, 2*(2), 174-207.

3. Proctor, R. W., & Vu, K. P. L. (2006). *Stimulus-response compatibility principles: Data, theory, and application.* CRC Press.

---

*心理学特講IIIA 第3回実習ノートブック | 担当：浅川伸一*